## CCA-F Lab 01: El Bucle Agéntico Canónico (Días 1 y 2)

### Objetivos de Aprendizaje para el Examen:
1. **API Stateless:** Comprender que la API de Anthropic no guarda memoria. Toda la conversación debe ser re-enviada acumulativamente en el arreglo `messages`.
2. **Control de Flujo por Contrato (`stop_reason`):** Aprender a gobernar el bucle exclusivamente con las señales estructurales de la API: `"tool_use"` y `"end_turn"`.
3. **Manejo de Roles:** Utilizar correctamente los roles en el historial: `user` (para el prompt inicial y los resultados de herramientas `tool_result`) y `assistant` (para los pensamientos y peticiones `tool_use` de Claude).

### ❌ Antipatrones Críticos evaluados en el Examen:
* **Natural Language Parsing:** Romper el ciclo buscando palabras clave en el texto de Claude como *"Terminé"*, *"Listo"* o *"Problema resuelto"*. (Falla porque Claude puede usar esas palabras en fases intermedias).
* **Text Content Check:** Asumir que si la respuesta tiene texto plano, el agente ya terminó. (Falla porque Claude puede responder texto descriptivo Y al mismo tiempo emitir un bloque de herramientas).
* **Iteration Cap Erróneo:** Usar un bucle rígido `for i in range(MAX)` como el *único* mecanismo de salida. (Se debe usar un ciclo `while True` gobernado por `stop_reason`, y el cap de iteraciones usarlo *únicamente* como una red de seguridad defensiva).

In [1]:
import os
import json
from anthropic import Anthropic
from dotenv import load_dotenv

# Cargar variables de entorno desde la raíz del proyecto
load_dotenv(dotenv_path="../.env")

# Verificar que la API key esté presente
if not os.environ.get("ANTHROPIC_API_KEY"):
    raise ValueError("⚠️ ANTHROPIC_API_KEY no encontrada en las variables de entorno.")

# Inicializar el cliente oficial de Anthropic
client = Anthropic()
print("✅ Cliente de Anthropic inicializado correctamente.")

✅ Cliente de Anthropic inicializado correctamente.


In [ ]:
# Definición de Herramientas del Sistema Simuladas (Mock Tools)
def get_system_status(service_name: str) -> dict:
    """Simula la verificación de logs y telemetría de un servicio de infraestructura."""
    services = {
        "database": {"status": "OPERATIONAL", "latency_ms": 14, "load_percentage": 42},
        "auth_service": {"status": "DEGRADED", "latency_ms": 1250, "load_percentage": 98},
        "payment_gateway": {"status": "OPERATIONAL", "latency_ms": 85, "load_percentage": 12}
    }
    return services.get(service_name.lower(), {"status": "UNKNOWN", "error": "Service not found"})

def restart_service(service_name: str) -> dict:
    """Simula un comando duro de reinicio de infraestructura para limpiar estados degradados."""
    return {
        "service": service_name,
        "action": "RESTART_EXECUTION",
        "status": "SUCCESS",
        "message": f"Service '{service_name}' was successfully restarted and telemetry re-initialized."
    }

# Esquemas de herramientas bajo el estándar JSON Schema exigido por Anthropic
TOOLS_SCHEMA = [
    {
        "name": "get_system_status",
        "description": "Retrieves the current operational status, telemetry metrics, load, and latency of a core infrastructure service.",
        "input_schema": {
            "type": "object",
            "properties": {
                "service_name": {
                    "type": "string",
                    "description": "The name of the service to check (e.g., 'database', 'auth_service', 'payment_gateway')."
                }
            },
            "required": ["service_name"]
        }
    },
    {
        "name": "restart_service",
        "description": "Executes an infrastructure restart command on a specific service to clear degraded states.",
        "input_schema": {
            "type": "object",
            "properties": {
                "service_name": {
                    "type": "string",
                    "description": "The target service name to be restarted."
                }
            },
            "required": ["service_name"]
        }
    }
]
print("🔧 Herramientas y esquemas definidos.")

🔧 Herramientas y esquemas definidos.


In [7]:
def run_agentic_loop(user_prompt: str):
    # Inicialización del State Ledger (Memoria acumulativa del agente)
    messages = [
        {"role": "user", "content": user_prompt}
    ]
    
    # Red de seguridad contra loops infinitos (Uso correcto del Iteration Cap)
    MAX_ITERATIONS = 5
    iteration = 0
    
    while True:
        iteration += 1
        if iteration > MAX_ITERATIONS:
            print("\n[⚠️ SAFETY BOUNDARY] Se alcanzó el límite preventivo de iteraciones.")
            break
            
        print(f"\n🚀 === INICIANDO ITERACIÓN {iteration} ===")
        
        # Llamada stateless a Claude 3.5 Sonnet
        response = client.messages.create(
            model="claude-sonnet-4-6", # Usamos Claude Sonnet 4.6 para esta prueba
            max_tokens=2000,
            system="You are an automated Site Reliability Engineer. Your goal is to diagnose and resolve system issues using tools.",
            tools=TOOLS_SCHEMA,
            messages=messages
        )
        
        # REGLA DE ORO DE ANTHROPIC: Registrar inmediatamente la respuesta entera del asistente
        messages.append({"role": "assistant", "content": response.content})
        
        # Procesar e imprimir lo que Claude ha devuelto en este turno
        for block in response.content:
            if block.type == "text":
                print(f"🤖 [Pensamiento de Claude]: {block.text}")
            elif block.type == "tool_use":
                print(f"🔧 [Claude solicita herramienta]: '{block.name}' con argumentos: {block.input}")

        # --- EVALUACIÓN DE CONTROL DE FLUJO SÓLIDO (Concepto Core del Examen) ---
        if response.stop_reason == "end_turn":
            print(f"\n🎯 [ÉXITO] Claude determinó que completó la tarea de forma limpia (stop_reason == 'end_turn').")
            break
            
        elif response.stop_reason == "tool_use":
            print("\n⚙️ Ejecutando herramientas solicitadas por el modelo...")
            
            # Iterar sobre los bloques de uso de herramientas (Preparación para llamadas concurrentes)
            for block in response.content:
                if block.type == "tool_use":
                    tool_name = block.name
                    tool_input = block.input
                    tool_use_id = block.id  # Identificador crítico e inmutable de la transacción
                    
                    # Enrutamiento determinista en nuestro backend
                    if tool_name == "get_system_status":
                        result_data = get_system_status(**tool_input)
                    elif tool_name == "restart_service":
                        result_data = restart_service(**tool_input)
                    else:
                        result_data = {"error": f"Tool '{tool_name}' not implemented."}
                        
                    print(f"📤 [Resultado de {tool_name}]: {result_data}")
                    
                    # REGLA DE ORO DE ANTHROPIC: Retornar el resultado estructurado bajo el rol 'user'
                    messages.append({
                        "role": "user",
                        "content": [
                            {
                                "type": "tool_result",
                                "tool_use_id": tool_use_id,
                                "content": json.dumps(result_data)
                            }
                        ]
                    })
        else:
            print(f"⚠️ [Manejo Alternativo] El loop terminó por una razón inusual: {response.stop_reason}")
            break

In [8]:
# Inyectamos el escenario de falla donde el auth_service requiere un análisis y posterior corrección
prompt_infraestructura = (
    "El sistema está reportando lentitud general. Por favor, revisa cómo está operando "
    "el servicio de autenticación ('auth_service') y, si encuentras que está fallando o degradado, "
    "ejecuta las acciones necesarias para corregirlo."
)

run_agentic_loop(prompt_infraestructura)


🚀 === INICIANDO ITERACIÓN 1 ===
🤖 [Pensamiento de Claude]: ¡Entendido! Voy a comenzar diagnosticando el estado actual del **auth_service**. 🔍
🔧 [Claude solicita herramienta]: 'get_system_status' con argumentos: {'service_name': 'auth_service'}

⚙️ Ejecutando herramientas solicitadas por el modelo...
📤 [Resultado de get_system_status]: {'status': 'DEGRADED', 'latency_ms': 1250, 'load_percentage': 98}

🚀 === INICIANDO ITERACIÓN 2 ===
🤖 [Pensamiento de Claude]: ⚠️ **Diagnóstico completado — Estado crítico detectado:**

| Métrica | Valor | Umbral Normal |
|---|---|---|
| 🔴 **Estado** | `DEGRADED` | `HEALTHY` |
| 🔴 **Latencia** | `1,250 ms` | `< 300 ms` |
| 🔴 **Carga** | `98%` | `< 80%` |

El **auth_service** está claramente degradado:
- La **latencia está 4x por encima** del umbral aceptable.
- La **carga está al límite máximo** (98%), lo que indica saturación del servicio.

Esto explica perfectamente la lentitud general del sistema, ya que la autenticación es una dependencia crítica para

Al ejecutar la aplicación, las 3 iteraciones que Claude realiza son:

- **Iteración 1:** Claude explica su plan, llama a get_system_status y se detiene en tool_use.

- **Iteración 2:** Claude ve los datos del sistema degradado (1250ms de latencia), deduce que necesita arreglarlo, llama a restart_service y vuelve a detenerse en tool_use.

- **Iteración 3:** Claude ve que el reinicio fue exitoso, escribe la conclusión final para el usuario y se detiene enviando un end_turn.